# 01: Data Collection

**Goal:** Download bill metadata from Congress.gov API and speeches from Stanford Congressional Record, then merge them.

## Steps
1. Set up Congress.gov API key
2. Fetch bills for Congress 110–114 (2007–2016)
3. Filter to economic subjects
4. Load Stanford speeches
5. Merge bills + speeches
6. Clean and save

**Data saved to:** `data/processed/bills_speeches_merged.csv`

In [ ]:
import sys
import os
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from src import data_utils

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Get Congress.gov API Key

**Instructions:**
1. Go to https://api.congress.gov
2. Register for a free account
3. Copy your API key
4. Paste it below OR set environment variable: `export CONGRESS_API_KEY=<key>`

In [ ]:
# Option 1: Retrieve from environment
try:
    api_key = data_utils.get_congress_gov_api_key()
    print(f"✓ API key found from environment")
except ValueError as e:
    print(f"Error: {e}")
    print("\nPlease set the CONGRESS_API_KEY environment variable or continue below.")
    # For testing, you can manually set here:
    # api_key = "YOUR_API_KEY_HERE"
    api_key = None

In [ ]:
# Option 2: Manually set API key (if env var not available)
# Uncomment and fill in your key:
# api_key = "PUT_YOUR_API_KEY_HERE"

# Check if we have a key
if api_key:
    print(f"✓ Ready to fetch data with API key: {api_key[:10]}...")
else:
    print("⚠ No API key found. Please set CONGRESS_API_KEY environment variable or uncomment the line above.")

## Step 2: Fetch Bills from Congress.gov

Download bill metadata for Congress 110–114, filtered to economic subjects.

In [ ]:
# Fetch bills for Congress 110–114
congress_numbers = [110, 111, 112, 113, 114]
all_bills = []

if api_key:
    for congress in congress_numbers:
        try:
            bills_df = data_utils.fetch_bills_from_congress_gov(congress, api_key)
            all_bills.append(bills_df)
            print(f"  Congress {congress}: {len(bills_df)} economic bills")
        except Exception as e:
            print(f"Error fetching Congress {congress}: {e}")
    
    # Combine all
    bills_combined = pd.concat(all_bills, ignore_index=True)
    print(f"\nTotal: {len(bills_combined)} bills across Congress 110–114")
else:
    print("Skipping API fetch. Please set API key above.")

In [ ]:
# Preview
if 'bills_combined' in locals():
    print(bills_combined.head())
    print(f"\nColumns: {bills_combined.columns.tolist()}")
    print(f"\nPass rate: {bills_combined['passed'].mean():.1%}")

## Step 3: Load Stanford Congressional Speeches

Load speeches from JSON files downloaded from https://data.stanford.edu/congress_text

**Expected file structure:** `data/raw/congress_110_speeches.json`, `congress_111_speeches.json`, etc.

In [ ]:
# Check if Stanford speech files exist
import os

raw_dir = "../data/raw"
if os.path.exists(raw_dir):
    files = os.listdir(raw_dir)
    speech_files = [f for f in files if "speech" in f.lower() or "congress_" in f.lower()]
    print(f"Files in {raw_dir}: {speech_files}")
    if not speech_files:
        print(f"\n⚠ No speech files found in {raw_dir}")
        print("Please download from https://data.stanford.edu/congress_text and save to data/raw/")
else:
    print(f"Directory {raw_dir} not found.")

In [ ]:
# Attempt to load speeches
try:
    speeches_df = data_utils.fetch_stanford_speeches([110, 111, 112, 113, 114], cache_dir="../data/raw")
    if len(speeches_df) > 0:
        print(f"Loaded {len(speeches_df)} speeches")
        print(f"\nColumns: {speeches_df.columns.tolist()}")
        print(f"\nSample:")
        print(speeches_df.head())
    else:
        print("No speeches loaded. Ensure Stanford files are in data/raw/")
except Exception as e:
    print(f"Error loading speeches: {e}")
    speeches_df = None

## Step 4: Merge Bills and Speeches

Match bill_id to link each bill with its floor speeches.

In [ ]:
# Merge if we have both datasets
if 'bills_combined' in locals() and speeches_df is not None and len(speeches_df) > 0:
    merged_df = data_utils.merge_bills_and_speeches(bills_combined, speeches_df)
    print(f"Merged dataset: {len(merged_df)} bills with speeches")
    print(f"\nColumns: {merged_df.columns.tolist()}")
else:
    print("Cannot merge: missing bills_combined or speeches_df")
    merged_df = None

In [ ]:
# Preview merged data
if merged_df is not None:
    print(merged_df[['bill_id', 'title', 'passed', 'speeches_combined']].head())
    print(f"\nPass rate in merged data: {merged_df['passed'].mean():.1%}")

## Step 5: Clean Data

Remove duplicates, nulls, and add derived features.

In [ ]:
# Clean
if merged_df is not None:
    cleaned_df = data_utils.clean_bill_speeches(merged_df)
    print(f"Cleaned dataset: {len(cleaned_df)} bills")
    print(f"\nNew columns: {cleaned_df.columns.tolist()}")
else:
    print("Skipping cleaning: no merged data")
    cleaned_df = None

In [ ]:
# Summary statistics
if cleaned_df is not None:
    print("\n=== Summary Statistics ===")
    print(f"Number of bills: {len(cleaned_df)}")
    print(f"Pass rate: {cleaned_df['passed'].mean():.1%}")
    print(f"\nBills by Congress:")
    print(cleaned_df['congress'].value_counts().sort_index())
    print(f"\nSpeech length (chars):")
    print(cleaned_df['speech_length'].describe())
    print(f"\nNumber of speakers per bill:")
    print(cleaned_df['num_speakers'].describe())

## Step 6: Save Processed Data

Save merged and cleaned data for downstream notebooks.

In [ ]:
# Save to CSV
if cleaned_df is not None:
    output_path = "../data/processed/bills_speeches_merged.csv"
    data_utils.save_processed_data(cleaned_df, output_path)
    print(f"\n✓ Data saved to {output_path}")
    print(f"  Shape: {cleaned_df.shape}")
else:
    print("Cannot save: no cleaned data")

## Summary

✓ Downloaded bill metadata from Congress.gov API  
✓ Loaded speeches from Stanford Congressional Record  
✓ Merged bills with speeches  
✓ Cleaned and saved to `data/processed/bills_speeches_merged.csv`

**Next:** Run `02_eda_preprocessing.ipynb`